# Import libraries

In [ ]:
import pandas as pd
import itertools
import math

# Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dict_llms = {
    1: ['openai'],
    2: ['llama'],
    3: ['moonshotai'],
    4: ['qwen'],
    5: ['deepseek']
}

In [ ]:
list_judges = ['openai', 'llama', 'moonshotai', 'qwen', 'tinyllama']

In [ ]:
input_path = 'drive/MyDrive/ViJudge/data/gen_judgement/'

# Process

In [ ]:
id_list = list(dict_llms.keys())
combinations = []
# Single: tạo các tập hợp [model_a, nan]
for combo in itertools.combinations(id_list, 1):
    combinations.append([combo[0], math.nan])
# Pair: tạo các tập hợp [model_a, model_b]
for combo in itertools.combinations(id_list, 2):
    combinations.append(list(combo))

## Gộp các tệp pair

In [ ]:
dict_file_single_paths = {}

for judge_name in list_judges:
    print(f"Processing pair files for LLM_judge: {judge_name}")
    list_file_pair_paths = []
    idx = 1
    
    for item in combinations:
        id_model_a = item[0]
        id_model_b = item[1]
        
        if isinstance(id_model_b, float) and math.isnan(id_model_b):
            file_path = input_path + judge_name + '/' + judge_name + 'Judge_single_' + dict_llms[id_model_a][0] + '.csv'
            key = (judge_name, idx)
            dict_file_single_paths[key] = file_path
            idx += 1
        else:
            file_path = input_path + judge_name + '/' + judge_name + 'Judge_pair_' + dict_llms[id_model_a][0] + '_' + dict_llms[id_model_b][0] + '.csv'
            list_file_pair_paths.append(file_path)

    pair_df = pd.DataFrame()
    for file_path in list_file_pair_paths:
        df = pd.read_csv(file_path)
        pair_df = pd.concat([pair_df, df], ignore_index=True)
        print(f"Processed file {file_path}")

    print('#'*30)
    pair_df.to_csv('drive/MyDrive/ViJudge/data/data_for_agreement/LLM_judge/' + judge_name + '_pair.csv', index=False)

## Kết hợp các tệp single thành pair_from_single

In [ ]:
for judge_name in list_judges:
    print(f"Judge: {judge_name}")
    idx = 1
    for item in combinations:
        id_model_a = item[0]
        id_model_b = item[1]

        if isinstance(id_model_b, float) and math.isnan(id_model_b):
            continue
        else:
            print(f"File pair {idx}:")

            file_path_a = dict_file_single_paths[(judge_name, id_model_a)]
            file_path_b = dict_file_single_paths[(judge_name, id_model_b)]
            df_a = pd.read_csv(file_path_a)
            df_b = pd.read_csv(file_path_b)

            merged_df = pd.DataFrame()
            question_ids = []
            for id in range(0, 100):
                new_id = 2000000 + id*10000 + id_model_a*100 + id_model_b
                question_ids.append(new_id)

            merged_df['question_id'] = question_ids
            merged_df['category'] = df_a['category']
            merged_df['model_a'] = df_a['model_a']
            merged_df['model_b'] = df_b['model_a']
            merged_df['conversation_a'] = df_a['conversation_a']
            merged_df['conversation_b'] = df_b['conversation_a']
            merged_df['judge_name'] = df_a['judge_name']
            merged_df['judge_type'] = ['pair_from_single'] * 100
            merged_df['judgement_explanation'] = 'A: ' + df_a['verdict'].astype(str) + ', B: ' + df_b['verdict'].astype(str)

            # Convert 'verdict' columns to numeric, coercing errors to NaN
            verdict_a_numeric = pd.to_numeric(df_a['verdict'], errors='coerce')
            verdict_b_numeric = pd.to_numeric(df_b['verdict'], errors='coerce')

            verdict_diff = verdict_a_numeric - verdict_b_numeric

            # Apply the A/B/C logic, handling NaN values as 'Unknown'
            merged_df['verdict'] = verdict_diff.apply(lambda x: 'A' if x > 0 else ('B' if x < 0 else ('C' if x == 0 else 'Unknown')))

            output_path = 'drive/MyDrive/ViJudge/data/data_for_agreement/LLM_judge/temp_pair_from_single/' + judge_name + '/' + judge_name + '_pair_' + dict_llms[id_model_a][0] + '_' + dict_llms[id_model_b][0] + '.csv'
            print(f"Processed file {output_path}")
            merged_df.to_csv(output_path, index=False)

            print(f"Completed file pair {idx}\n")
            idx += 1
    print('#'*30)

## Gộp các tệp pair_from_single

In [ ]:
file_path_term = 'drive/MyDrive/ViJudge/data/data_for_agreement/LLM_judge/temp_pair_from_single/'
output_pair_from_single_path = 'drive/MyDrive/ViJudge/data/data_for_agreement/LLM_judge/'

for judge_name in list_judges:
    print(f"Processing pair files for LLM_judge: {judge_name}")
    list_file_pair_paths = []
    idx = 1
    
    for item in combinations:
        id_model_a = item[0]
        id_model_b = item[1]
        
        if isinstance(id_model_b, float) and math.isnan(id_model_b):
            continue
        else:
            file_path = file_path_term + judge_name + '/' + judge_name + '_pair_' + dict_llms[id_model_a][0] + '_' + dict_llms[id_model_b][0] + '.csv'
            list_file_pair_paths.append(file_path)

    pair_df = pd.DataFrame()
    for file_path in list_file_pair_paths:
        df = pd.read_csv(file_path)
        pair_df = pd.concat([pair_df, df], ignore_index=True)
        print(f"Merged file {file_path}")

    print('#'*30)
    pair_df.to_csv(output_pair_from_single_path + judge_name + '_pair_from_single.csv', index=False)